# FinChain GRPO RunPod Notebook

GRPO is the headline RLVR method for this repo: sample a group of completions per prompt, score each with the FinChain verifier, normalize rewards within the group, and update the policy with KL control. This notebook uses TRL for the production trainer and keeps the repo math in `src/finpost/posttraining/grpo.py` as the readable reference.

## RunPod Terminal Preflight
Run this in a JupyterLab terminal before opening the notebook cells:

```bash
cd /workspace
if [ ! -d finpost ]; then
  git clone https://github.com/shannan-liu1/finpost.git
fi
cd /workspace/finpost
git pull --ff-only
pip install -e ".[dev,rlvr]"
python -c "import finpost, torch; print('finpost ok'); print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')"

export FINPOST_FINCHAIN_TRAIN_JSONL=/workspace/data/finchain/train.jsonl
export FINPOST_FINCHAIN_VALIDATION_JSONL=/workspace/data/finchain/validation.jsonl
export FINPOST_FINCHAIN_TEST_JSONL=/workspace/data/finchain/test.jsonl
export WANDB_MODE=offline
```

If `import finpost` fails after `pip install -e`, use the fallback from `docs/runbooks/runpod-bootstrap.md`:

```bash
echo "/workspace/finpost/src" > /usr/local/lib/python3.11/dist-packages/finpost.pth
python -c "import finpost; print(finpost.__file__)"
```

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

REPO = Path('/workspace/finpost') if Path('/workspace/finpost').exists() else Path.cwd()
os.chdir(REPO)

MODEL_ID = 'Qwen/Qwen2.5-1.5B'
SFT_CONFIG = 'experiments/finchain/qwen25_1_5b_sft_runpod.yaml'
SFT_RUN_ROOT = Path('results/checkpoints/qwen25-1p5b-finchain-sft')
SFT_HF_DIR = Path('results/checkpoints/qwen25-1p5b-finchain-sft-hf')
DPO_CONFIG = 'experiments/dpo/finchain_qwen25_1_5b.yaml'
PAIR_RUN = Path('results/finchain_pairs/run_001')

os.environ.setdefault('WANDB_MODE', 'offline')


def run(cmd: str, *, check: bool = True) -> subprocess.CompletedProcess:
    print('$', cmd)
    return subprocess.run(cmd, shell=True, check=check, text=True)

print('repo:', REPO)
print('WANDB_MODE:', os.environ.get('WANDB_MODE'))

In [ ]:
import torch

print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu count:', torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 1), 'GB')
else:
    print('No CUDA GPU visible. Stop here on RunPod.')

In [ ]:
from finpost.data.finchain import load_finchain, resolve_finchain_path

for split in ['train', 'validation', 'test']:
    path = resolve_finchain_path(split)
    print(split, '->', path, 'exists=', path.exists())

train_examples = load_finchain('train')
print('train examples:', len(train_examples))
print('first prompt id:', train_examples[0].id)
print('first gold answer:', train_examples[0].final_answer)

## Why Use TRL Here
The from-scratch GRPO primitive is valuable for understanding: group advantages, token log-probs, old-policy ratios, and KL. But the expensive system work is rollout scheduling, distributed launch, memory handling, and logging. TRL implements that trainer interface, and the repo supplies the FinChain reward function.

In [ ]:
from finpost.posttraining.finchain_rlvr import finchain_binary_rewards

print(finchain_binary_rewards(['Final Answer: 10', 'Final Answer: 11'], gold_answer=['10', '10']))

## Single-GPU GRPO Canary
Run a tiny canary before spending real money. The defaults are intentionally conservative for one A40.

```bash
cd /workspace/finpost
python scripts/train_finchain_trl_grpo.py \
  --model Qwen/Qwen2.5-1.5B \
  --train-n 256 \
  --max-steps 20 \
  --num-generations 4 \
  --max-completion-length 512 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 8 \
  --output-dir results/checkpoints/qwen25-1p5b-finchain-grpo-canary
```

In [ ]:
single_gpu_grpo = f"""
python scripts/train_finchain_trl_grpo.py \
  --model {MODEL_ID} \
  --train-n 256 \
  --max-steps 20 \
  --num-generations 4 \
  --max-completion-length 512 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 8 \
  --output-dir results/checkpoints/qwen25-1p5b-finchain-grpo-canary
""".strip()
print(single_gpu_grpo)

## Two-GPU GRPO
Use Accelerate for real multi-GPU training. Start without vLLM; add `--use-vllm` only after the non-vLLM canary works, because vLLM adds another moving part.

```bash
cd /workspace/finpost
accelerate launch --num_processes 2 scripts/train_finchain_trl_grpo.py \
  --model Qwen/Qwen2.5-1.5B \
  --train-n 2000 \
  --max-steps 300 \
  --num-generations 4 \
  --max-completion-length 512 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 8 \
  --output-dir results/checkpoints/qwen25-1p5b-finchain-grpo-2gpu
```

If generation is clearly the bottleneck after the canary, install the optional vLLM extra and run with `--use-vllm`:

```bash
pip install -e ".[vllm]"

accelerate launch --num_processes 2 scripts/train_finchain_trl_grpo.py \
  --model Qwen/Qwen2.5-1.5B \
  --train-n 2000 \
  --max-steps 300 \
  --num-generations 4 \
  --max-completion-length 512 \
  --use-vllm \
  --vllm-gpu-memory-utilization 0.25 \
  --output-dir results/checkpoints/qwen25-1p5b-finchain-grpo-2gpu-vllm
```

In [ ]:
two_gpu_grpo = f"""
accelerate launch --num_processes 2 scripts/train_finchain_trl_grpo.py \
  --model {MODEL_ID} \
  --train-n 2000 \
  --max-steps 300 \
  --num-generations 4 \
  --max-completion-length 512 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 8 \
  --output-dir results/checkpoints/qwen25-1p5b-finchain-grpo-2gpu
""".strip()
print(two_gpu_grpo)

## Evaluation And Exit Gate
Evaluate against the same held-out FinChain split as SFT/DPO/OPD.

```bash
python scripts/run_finchain_eval.py \
  --checkpoints grpo=results/checkpoints/qwen25-1p5b-finchain-grpo-2gpu/final \
  --n 500 \
  --out-dir results/evals/finchain_grpo_2gpu \
  --device cuda
```

Keep GRPO if it improves held-out final-answer accuracy or pass@K per dollar. Stop the run early if parseability rises while correctness is flat, if KL explodes, or if completions collapse into a template that fools the parser.